<a href="https://colab.research.google.com/github/shrutikhati70-hash/Datawarehouisng_and_bigdata_analytics/blob/main/Map_Reduce" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1.Load Dataset

In [3]:
import pandas as pd


from google.colab import files
uploaded = files.upload()

file_name = "previous_semester_final_exam_results.csv"
df = pd.read_csv("previous_semester_final_exam_results.csv")

print("Dataset shape:", df.shape)
display(df.head(10))

Saving previous_semester_final_exam_results.csv to previous_semester_final_exam_results (1).csv
Dataset shape: (150, 3)


,Student_ID,Subject,Grade
0,S01,Mathematics,S
1,S02,Mathematics,A
2,S03,Mathematics,B
3,S04,Mathematics,C
4,S05,Mathematics,D
5,S06,Mathematics,E
6,S07,Mathematics,F
7,S08,Mathematics,A
8,S09,Mathematics,B
9,S10,Mathematics,C


##2.Check and Clean the Data

In [4]:
valid_grades = ["S", "A", "B", "C", "D", "E", "F"]

df["Student_ID"] = df["Student_ID"].astype(str).str.strip()
df["Subject"] = df["Subject"].astype(str).str.strip()
df["Grade"] = df["Grade"].astype(str).str.strip().str.upper()

# Keep only valid grade categories
df = df[df["Grade"].isin(valid_grades)].copy()

print("Students:", df["Student_ID"].nunique())
print("Subjects:", df["Subject"].nunique())
print("Exam records:", len(df))
print("Grade categories:", sorted(df["Grade"].unique()))
display(df.head())

Students: 30
Subjects: 5
Exam records: 150
Grade categories: ['A', 'B', 'C', 'D', 'E', 'F', 'S']


,Student_ID,Subject,Grade
0,S01,Mathematics,S
1,S02,Mathematics,A
2,S03,Mathematics,B
3,S04,Mathematics,C
4,S05,Mathematics,D


##3.Map Phase

In [5]:
mapped = [
    ((row.Subject, row.Grade), 1)
    for row in df.itertuples(index=False)
]

print("First 15 mapper outputs:")
for item in mapped[:15]:
    print(item)

First 15 mapper outputs:
(('Mathematics', 'S'), 1)
(('Mathematics', 'A'), 1)
(('Mathematics', 'B'), 1)
(('Mathematics', 'C'), 1)
(('Mathematics', 'D'), 1)
(('Mathematics', 'E'), 1)
(('Mathematics', 'F'), 1)
(('Mathematics', 'A'), 1)
(('Mathematics', 'B'), 1)
(('Mathematics', 'C'), 1)
(('Mathematics', 'A'), 1)
(('Mathematics', 'B'), 1)
(('Mathematics', 'C'), 1)
(('Mathematics', 'D'), 1)
(('Mathematics', 'A'), 1)


##4.Readuce Phase

In [6]:
reduced = {}

for key, value in mapped:
    reduced[key] = reduced.get(key, 0) + value

print("Reducer output:")
for key in sorted(reduced):
    print(key, "->", reduced[key])

Reducer output:
('Computer Networks', 'A') -> 6
('Computer Networks', 'B') -> 6
('Computer Networks', 'C') -> 6
('Computer Networks', 'D') -> 6
('Computer Networks', 'E') -> 4
('Computer Networks', 'F') -> 1
('Computer Networks', 'S') -> 1
('Data Structures', 'A') -> 8
('Data Structures', 'B') -> 6
('Data Structures', 'C') -> 6
('Data Structures', 'D') -> 5
('Data Structures', 'E') -> 3
('Data Structures', 'F') -> 1
('Data Structures', 'S') -> 1
('Database Management', 'A') -> 7
('Database Management', 'B') -> 6
('Database Management', 'C') -> 6
('Database Management', 'D') -> 6
('Database Management', 'E') -> 3
('Database Management', 'F') -> 1
('Database Management', 'S') -> 1
('Mathematics', 'A') -> 7
('Mathematics', 'B') -> 7
('Mathematics', 'C') -> 6
('Mathematics', 'D') -> 5
('Mathematics', 'E') -> 3
('Mathematics', 'F') -> 1
('Mathematics', 'S') -> 1
('Operating Systems', 'A') -> 7
('Operating Systems', 'B') -> 6
('Operating Systems', 'C') -> 6
('Operating Systems', 'D') -> 6
('

##5.Final result:Number of student in each category for every subject

In [7]:
result = pd.DataFrame(
    [
        {"Subject": subject, "Grade": grade, "Student_Count": count}
        for (subject, grade), count in reduced.items()
    ]
)

pivot_result = (
    result
    .pivot(index="Subject", columns="Grade", values="Student_Count")
    .fillna(0)
    .astype(int)
    .reindex(columns=valid_grades, fill_value=0)
)

display(pivot_result)

Grade,S,A,B,C,D,E,F
Subject,,,,,,,
Computer Networks,1,6,6,6,6,4,1
Data Structures,1,8,6,6,5,3,1
Database Management,1,7,6,6,6,3,1
Mathematics,1,7,7,6,5,3,1
Operating Systems,1,7,6,6,6,3,1


##6.Overall grade count across all subjects

In [8]:
overall = (
    df["Grade"]
    .value_counts()
    .reindex(valid_grades, fill_value=0)
    .rename_axis("Grade")
    .reset_index(name="Student_Count")
)

display(overall)

,Grade,Student_Count
0,S,5
1,A,35
2,B,31
3,C,30
4,D,28
5,E,16
6,F,5


##7.Verification

In [9]:
subject_totals = df.groupby("Subject")["Student_ID"].nunique()

print("Students per subject:")
display(subject_totals.to_frame("Number_of_Students"))

assert (subject_totals == 30).all()
print("Verification passed: every subject has 30 students.")

Students per subject:


,Number_of_Students
Subject,
Computer Networks,30
Data Structures,30
Database Management,30
Mathematics,30
Operating Systems,30


Verification passed: every subject has 30 students.


Conclusion
In this assignment, I used the MapReduce approach to count the number of students in each grade category for every subject. In the Map phase, each record was converted into a subject-grade pair with a value of 1. In the Reduce phase, the values with the same key were added.

The final table shows the grade-wise count for each subject, and the verification step checks that the subject-wise student count is 30.